# Exploratory data analysis
Load the feature table built by `prepare_data.py`.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

X = pd.read_parquet("../data/processed/train_features.parquet")
print(X.shape)
X.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/train_features.parquet'

## How rare are defaults?
This is why accuracy is useless here: predicting 'never defaults' is already ~92% accurate.

In [ ]:
print(f"Default rate: {X['TARGET'].mean():.2%}")
X["TARGET"].value_counts()

## Missing values
Bureau and history features are missing for thin-file applicants. XGBoost handles NaN natively.

In [ ]:
X.drop(columns="TARGET").isna().mean().sort_values(ascending=False).round(3).head(15)

## Default rate by band
Each feature is split into 5 equal-sized groups. A good feature shows a clear trend.

In [ ]:
for col in ["credit_to_income", "payment_to_income", "employment_years", "late_payment_ratio", "ext_source_mean"]:
    bands = pd.qcut(X[col], q=5, duplicates="drop")
    rate = X.groupby(bands, observed=True)["TARGET"].mean().round(3)
    print(f"\nDefault rate by {col}:\n{rate.to_string()}")

## Default rate by category

In [ ]:
for col in ["education", "income_type", "housing_type"]:
    print(X.groupby(col, observed=True)["TARGET"].agg(["mean", "count"]).round(3).sort_values("mean"), "\n")

## Distributions: repaid vs defaulted
The more the two histograms differ, the more useful the feature.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["ext_source_mean", "credit_to_income", "age_years"]):
    X[X.TARGET == 0][col].plot.hist(bins=50, alpha=0.5, density=True, ax=ax, label="repaid")
    X[X.TARGET == 1][col].plot.hist(bins=50, alpha=0.5, density=True, ax=ax, label="defaulted")
    ax.set_title(col); ax.legend()
plt.tight_layout()

## Correlation with TARGET
Positive = higher value, more risk. Negative = higher value, less risk.

In [ ]:
X.drop(columns="TARGET").corrwith(X["TARGET"], numeric_only=True).sort_values().round(3)